# Beyond Transformers And World Models

The earlier attention notebook derived the `T x T` score matrix directly from queries and keys. This notebook uses that derivation as a reference point for reading architecture papers that try to relax, route around, or complement vanilla self-attention.

The goal is orientation, not implementation. We are mapping the pressure points: long-context cost, memory movement, inductive bias, training objective, and what evidence a paper actually provides.

## 1. The vanilla attention bottleneck

Full causal self-attention computes a similarity score between each token and every earlier token. Per head, that creates a score matrix with roughly `T^2` entries. The earlier derivation matters because every proposed replacement should be compared against that baseline: what matrix is still materialized, what interactions are dropped, and what hardware pattern changes.

Two separate costs get mixed together in casual discussion:
- **Score count:** how many pairwise interactions are computed.
- **Memory traffic:** how many keys, values, and cache entries must be stored and read.

Some methods mainly reduce score count. Others leave score complexity alone but reduce cache pressure or improve the constant factors during decoding.

In [ ]:
sequence_lengths = [128, 512, 2048, 8192]
quadratic_scores = {n: n * n for n in sequence_lengths}
linear_window_scores = {n: n * 256 for n in sequence_lengths}
quadratic_scores, linear_window_scores

## 2. Sparse and local attention

Sparse attention starts from a blunt question: does every token really need to inspect every previous token? If a task mostly depends on nearby context, a model can restrict each token to a local neighborhood and compute far fewer scores.

Locality introduces an inductive bias. That can help when nearby structure dominates, but it can also hide useful long-range relationships unless the pattern includes occasional routes for distant information.

## 3. Sliding-window, block, and global-token patterns

A sliding window keeps each token connected to only the most recent `w` positions. For long sequences, that gives rough score growth of `T * w` instead of `T^2`. For short sequences where `T <= w`, the local window cannot be more expensive than full attention because the window simply covers the whole available prefix, so a better finite-length estimate is `T * min(T, w)`. Block patterns partition the sequence into chunks so attention is dense inside a block and sparse across blocks. Global-token patterns designate a small set of tokens that can communicate broadly, acting as routing hubs.

| Pattern | Rough score count per head | Main idea | Typical tradeoff |
| --- | --- | --- | --- |
| Full attention | `T^2` | Every token can score against every earlier token | Expensive at long context lengths |
| Sliding-window attention | `T * min(T, w)` and about `T * w` once `T >> w` | Keep only nearby interactions inside a fixed window | Long-range links require many hops |
| Block sparse attention | About `num_blocks * block_size^2` plus cross-block links | Dense local blocks with a chosen routing pattern | Pattern design matters as much as asymptotics |
| Global-token hybrids | Mostly local plus a small global pathway | Preserve a few broadcast or aggregation channels | Global token choice becomes a modeling assumption |

The key reading habit is to ask whether the sparsity pattern matches the data structure or merely improves an asymptotic headline.

## 4. Grouped-query and multi-query attention

Grouped-query attention (GQA) and multi-query attention (MQA) do **not** primarily attack the `T^2` score matrix. Their practical win is different: they share keys and values across multiple query heads, which reduces KV-cache size and memory bandwidth during autoregressive decoding.

That distinction matters because a paper can improve generation throughput without changing the fundamental score complexity of prefill attention. When reading such a claim, separate **training-time attention cost** from **generation-time cache cost**.

In [ ]:
sequence_lengths = [128, 512, 2048, 8192]
window = 256
comparison_rows = []
for n in sequence_lengths:
    full = n * n
    windowed = n * min(n, window)
    comparison_rows.append((n, full, windowed, full / windowed))
comparison_rows

## 5. Subquadratic attention ambitions

Subquadratic attention papers aim for growth better than `T^2` while preserving enough expressive power to stay useful. The mechanisms vary: kernel approximations, low-rank structure, learned routing, hashing, compression, or hybrid retrieval patterns.

At this curriculum stage, the important habit is not memorizing families of methods. Instead, ask four durable questions:
1. What exact quantity becomes subquadratic: score count, memory, cache reads, or wall-clock time?
2. What approximation or structural assumption makes that possible?
3. Which workloads benefit: long-context prefill, decoding, or both?
4. What capability might degrade when interactions are no longer fully dense?

## 6. Recurrence and state-space alternatives

Another path is to step outside attention-heavy designs entirely. Recurrent models and state-space models maintain a compressed state that updates over time instead of materializing a full pairwise score matrix.

Conceptually, this changes the question from 'which previous tokens should I score against?' to 'what running state should summarize what matters so far?' That can be attractive for long sequences, streaming inputs, or settings where constant-memory updates matter more than unrestricted pairwise access.

## 7. Predictive representation learning

Post-transformer exploration is not only about architecture. It is also about objective design. Autoregressive language models predict the next token directly. Predictive representation learning instead asks a model to predict useful future **representations** or latent targets.

This changes what the system is encouraged to store. Rather than preserving whatever is needed for exact next-token recovery, the model can focus on compact state that is predictive of future structure at a higher level of abstraction.

## 8. JEPA-style objective intuition

JEPA-style objectives are an orientation topic here, not an implementation target. The high-level intuition is that a model observes part of a scene, sequence, or trajectory and learns a latent representation that predicts another latent representation for a related future or missing part.

That differs from next-token generation in two useful ways:
- The target can live in a learned representation space rather than in raw token or pixel space.
- The objective can emphasize semantic predictability over exact low-level reconstruction.

When you read JEPA-like work, track what is predicted, in what representation space, and what invariances the objective is trying to preserve.

## 9. World models, memory, planning, grounding, and agency

World-model discussions push beyond short-horizon sequence continuation toward systems that maintain durable state, represent environment dynamics, and support planning or grounded interaction.

This notebook stays at the framing level:
- **Memory:** what information persists beyond a single context window?
- **Planning:** does the model evaluate possible futures or only continue the next step?
- **Grounding:** are the targets tied to text alone, or to actions, observations, or other modalities?
- **Agency:** does the system only predict, or does it act and receive feedback from an environment?

Those questions point toward later research areas without requiring us to build agents or environment simulators in this curriculum task.

## 10. How to read future architecture papers

A good reading loop is:
1. Anchor on the baseline: compare every claim to the vanilla `T x T` attention picture from Notebook 4.
2. Identify the bottleneck: is the paper targeting compute, memory, latency, supervision, or evaluation weakness?
3. Identify the inductive bias: what structure is assumed about locality, recurrence, hierarchy, or latent state?
4. Separate asymptotics from systems reality: a better big-O claim may still lose on hardware if memory movement or kernel efficiency gets worse.
5. Inspect the evidence: what benchmark or workload actually demonstrates the claimed advantage?

The companion reading map in `docs/notes/beyond_transformers_reading_map.md` turns those questions into a reusable checklist for future papers.

## Exercise checkpoint

Use the companion exercise file to rehearse the main distinctions:
- why `T^2` versus `T * w` matters,
- why GQA changes KV-cache pressure more than score complexity,
- why post-transformer work can change the objective as much as the architecture, and
- how to classify a paper's claimed improvement before accepting its headline.